
The objective of this notebook is to evaluate how changes in demand and capacity affect network cost, facility utilization, and operational feasibility.

Sensitivity analysis helps decision-makers understand which network components are most critical to performance.

In [ ]:
# Import optimization and analysis libraries

import pandas as pd
import numpy as np

from pulp import (
    LpProblem,
    LpMinimize,
    LpVariable,
    lpSum,
    LpStatus,
    value
)

file_path = "../data/raw/Supply chain logistics problem.xlsx"

order_list = pd.read_excel(file_path, sheet_name="OrderList")
freight_rates = pd.read_excel(file_path, sheet_name="FreightRates")
wh_costs = pd.read_excel(file_path, sheet_name="WhCosts")
wh_capacities = pd.read_excel(file_path, sheet_name="WhCapacities")
products_per_plant = pd.read_excel(file_path, sheet_name="ProductsPerPlant")
vmi_customers = pd.read_excel(file_path, sheet_name="VmiCustomers")
plant_ports = pd.read_excel(file_path, sheet_name="PlantPorts")

    plant_summary = pd.read_csv(
        "../data/processed/plant_summary.csv"
    )

    customer_demand_scaled = pd.read_csv(
        "../data/processed/customer_demand_scaled.csv"
    )

In [12]:
# Solve transportation optimization scenario

def solve_network_scenario(
    demand_dict,
    capacity_dict,
    cost_dict,
    scenario_name="Scenario"
):

    model = LpProblem(
        scenario_name,
        LpMinimize
    )

    x = LpVariable.dicts(
        "shipment",
        [(i, j) for i in plants for j in customers],
        lowBound=0
    )

    # Objective
    model += lpSum(
        cost_dict[i] * x[(i, j)]
        for i in plants
        for j in customers
    )

    # Demand constraints
    for j in customers:
        model += (
            lpSum(
                x[(i, j)]
                for i in plants
            )
            >= demand_dict[j]
        )

    # Capacity constraints
    for i in plants:
        model += (
            lpSum(
                x[(i, j)]
                for j in customers
            )
            <= capacity_dict[i]
        )

    model.solve()

    return {
        "Scenario": scenario_name,
        "Status": LpStatus[model.status],
        "Cost": value(model.objective)
    }

In [13]:
# Plant list
plants = plant_summary['Plant ID'].tolist()

# Customer list
customers = customer_demand_scaled['Customer'].tolist()

# Capacity dictionary
capacity = dict(
    zip(
        plant_summary['Plant ID'],
        plant_summary['Daily Capacity ']
    )
)

# Warehouse cost dictionary
warehouse_cost_dict = dict(
    zip(
        plant_summary['Plant ID'],
        plant_summary['Cost/unit']
    )
)

# Demand dictionary
demand = dict(
    zip(
        customer_demand_scaled['Customer'],
        customer_demand_scaled['Scaled_Demand']
    )
)

In [14]:
baseline_result = solve_network_scenario(
    demand,
    capacity,
    warehouse_cost_dict,
    "Baseline"
)

baseline_result

{'Scenario': 'Baseline', 'Status': 'Optimal', 'Cost': 3300.7480749470974}

In [15]:
demand_10 = {
    k: v * 1.10
    for k, v in demand.items()
}

scenario_a = solve_network_scenario(
    demand_10,
    capacity,
    warehouse_cost_dict,
    "Demand +10%"
)

scenario_a

f:\Projects\transport-network-optimization\.venv\Lib\site-packages\pulp\pulp.py:1706: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


{'Scenario': 'Demand +10%', 'Status': 'Infeasible', 'Cost': 4167.002616658402}

In [16]:
demand_20 = {
    k: v * 1.20
    for k, v in demand.items()
}

scenario_b = solve_network_scenario(
    demand_20,
    capacity,
    warehouse_cost_dict,
    "Demand +20%"
)

scenario_b

{'Scenario': 'Demand +20%', 'Status': 'Infeasible', 'Cost': 4670.1958796425415}

In [17]:
capacity_c = capacity.copy()

capacity_c['PLANT01'] *= 0.75

scenario_c = solve_network_scenario(
    demand,
    capacity_c,
    warehouse_cost_dict,
    "PLANT01 -25%"
)

scenario_c

{'Scenario': 'PLANT01 -25%', 'Status': 'Optimal', 'Cost': 3672.988354682148}

In [18]:
capacity_d = capacity.copy()

capacity_d['PLANT03'] = 0

scenario_d = solve_network_scenario(
    demand,
    capacity_d,
    warehouse_cost_dict,
    "PLANT03 Outage"
)

scenario_d


{'Scenario': 'PLANT03 Outage',
 'Status': 'Infeasible',
 'Cost': 4008.788038366268}

In [19]:
scenario_results = pd.DataFrame([
    baseline_result,
    scenario_a,
    scenario_b,
    scenario_c,
    scenario_d
])

scenario_results

,Scenario,Status,Cost
0,Baseline,Optimal,3300.748075
1,Demand +10%,Infeasible,4167.002617
2,Demand +20%,Infeasible,4670.195880
3,PLANT01 -25%,Optimal,3672.988355
4,PLANT03 Outage,Infeasible,4008.788038


The network became infeasible under both 10% and 20% demand growth scenarios.
This indicates that the baseline network operates with very limited excess capacity and cannot absorb moderate increases in demand without additional supply resources.

### PLANT01 Capacity Reduction
Reducing PLANT01 capacity by 25% increased total network cost by approximately 11%.
The network remained feasible, but required greater utilization of higher-cost facilities.

### PLANT03 Outage
Removing PLANT03 resulted in an infeasible solution.
This indicates that PLANT03 is a critical supply node whose capacity cannot be replaced by the remaining facilities.

### Strategic Implications
The network is highly dependent on a small number of high-capacity plants.

In [20]:
total_capacity = sum(capacity.values())
total_demand = sum(demand.values())

reserve_capacity = total_capacity - total_demand

print("Total Capacity:", total_capacity)
print("Total Demand:", total_demand)
print("Reserve Capacity:", reserve_capacity)
print("Reserve Capacity %:", reserve_capacity / total_capacity * 100)

Total Capacity: 5791
Total Demand: 5501.45
Reserve Capacity: 289.5500000000002
Reserve Capacity %: 5.000000000000003


The baseline network contains only 289.55 units of reserve capacity, representing approximately 5% of total available capacity.
This indicates that the network operates at roughly 95% utilization under normal conditions.
The limited reserve capacity explains why the network became infeasible under a 10% demand growth scenario.